# L5 · Notebook 02 — 步长条件失败案例

**对应 PPT**：`L5-SA.tex` ★ 两条步长条件的两面性（line 284）

## 教学目标

Robbins-Monro 收敛定理要求 (C1) 步长两条件：
$$\sum_k \alpha_k = \infty \quad\text{(走得够远)} \qquad \sum_k \alpha_k^2 < \infty \quad\text{(噪声累积可控)}$$

本 notebook 通过**双反例**直观看清两条件各自的作用：

1. **破坏 $\sum\alpha=\infty$**：$\alpha_k=1/k^2$，$\sum\alpha_k=\pi^2/6\approx 1.64$ 有限 $\Rightarrow$ 序列**卡住**（永远走不到 $\mu^*$）
2. **破坏 $\sum\alpha^2<\infty$**：$\alpha_k=0.3$（常数），$\sum\alpha_k^2=\infty$ $\Rightarrow$ 序列在 $\mu^*$ 附近**永远抖动**（不收敛到点）
3. **同时满足**：$\alpha_k=1/(k+1)$ $\Rightarrow$ 几乎必然收敛

## 任务

估计 $\mu = \E[X]$，其中 $X\sim\mathcal N(5, 1)$。RM 迭代：$\mu_{k+1} = \mu_k + \alpha_k(x_{k+1} - \mu_k)$。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng_seed = 0
mu_true = 5.0
n_steps = 1000

def run_rm(alpha_fn, mu0=0.0, seed=rng_seed):
    rng = np.random.default_rng(seed)
    mu = mu0
    history = [mu]
    for k in range(n_steps):
        x = rng.normal(mu_true, 1.0)
        a = alpha_fn(k)
        mu = mu + a * (x - mu)
        history.append(mu)
    return np.array(history)

schemes = {
    '1/k (合规)':          lambda k: 1.0 / (k + 1),
    '0.3 常数 (破 sum a²<∞)': lambda k: 0.3,
    '0.05/k² (破 sum a=∞)':  lambda k: 0.05 / (k + 1)**2,
}

trajectories = {name: run_rm(fn) for name, fn in schemes.items()}

print(f'真值 μ* = {mu_true}')
print(f"{'步长':<25} | μ_final  | |μ_final-μ*|")
for name, traj in trajectories.items():
    err = abs(traj[-1] - mu_true)
    print(f'{name:<25} | {traj[-1]:7.4f}  | {err:7.4f}')

## 1. 三种步长的轨迹对比

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = {'1/k (合规)': 'C2', '0.3 常数 (破 sum a²<∞)': 'C1', '0.05/k² (破 sum a=∞)': 'C3'}
for name, traj in trajectories.items():
    ax.plot(traj, label=name, color=colors[name], linewidth=1.2, alpha=0.85)
ax.axhline(mu_true, color='black', linestyle='--', linewidth=0.8, label=f'$\\mu^* = {mu_true}$')
ax.set_xlabel('iteration k')
ax.set_ylabel(r'$\mu_k$')
ax.set_title('RM step-size: 合规 vs 破坏 (C1.1) vs 破坏 (C1.2)')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/step_size_failure.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. 累积步长 vs 累积平方步长（验证哪条被破坏）

把 $\sum_{k\le K}\alpha_k$ 与 $\sum_{k\le K}\alpha_k^2$ 各自画出来，看它们随 $K$ 是否发散。

In [ ]:
k_range = np.arange(n_steps)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for name, fn in schemes.items():
    alphas = np.array([fn(int(k)) for k in k_range])
    axes[0].plot(np.cumsum(alphas), label=name, color=colors[name])
    axes[1].plot(np.cumsum(alphas**2), label=name, color=colors[name])

axes[0].set_xlabel('K'); axes[0].set_ylabel(r'$\sum_{k=0}^K \alpha_k$')
axes[0].set_title(r'累积步长 (条件 1：应 $\to \infty$)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].set_xlabel('K'); axes[1].set_ylabel(r'$\sum_{k=0}^K \alpha_k^2$')
axes[1].set_title(r'累积平方步长 (条件 2：应 $< \infty$)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/step_size_cumulative.png', dpi=120, bbox_inches='tight')
plt.show()

# 数值核对
for name, fn in schemes.items():
    alphas = np.array([fn(int(k)) for k in k_range])
    print(f'{name:<25} | sum α = {alphas.sum():9.4f}  | sum α² = {(alphas**2).sum():9.4f}')

## 3. 现象诊断

| 步长 | $\sum\alpha$ | $\sum\alpha^2$ | 结果 |
|---|---|---|---|
| $1/(k+1)$ | $\to\infty$（调和） | $\to \pi^2/6\approx 1.64$ | **收敛**到 $\mu^*$ |
| $0.3$ 常数 | $\to\infty$ | $\to\infty$ | **抖动**（围绕 $\mu^*$ 永远振荡）|
| $1/(k+1)^2$ | $\to\pi^2/6\approx 1.64$（有限）| $\to\pi^4/90\approx 1.08$ | **卡住**（累计移动有限 $\Rightarrow$ 从 0 走不到 5）|

## 工程意义

- **典型选择 $\alpha_k = 1/k^p, p\in(1/2, 1]$**：同时满足两条件
- **实务中常用 $\alpha=$ 常数**（深度 RL，DQN/PPO 都这样）：放弃严格收敛，换更快的近 $\mu^*$ 速度
- **学习率衰减**（如 cosine annealing）的本质就是把 $\alpha$ 从大到小调，逼近 RM 第二条件

## 思考题

1. 试 $\alpha_k = 1/(k+1)^{0.5}$，是否满足两条件？为什么仍能收敛？
2. 把 0.3 常数换成 $\alpha_k = 0.3 \cdot 0.99^k$（指数衰减），是否同时满足两条件？
3. 这与 L7 TD(0) 中的步长选择有何关联？（提示：episodic 任务常用 1/n_visits）